# Distillation RL-Assisted MPC Weight Multipliers

This notebook uses the shared weight-multiplier runner and plotting stack, but the default experiment settings are aligned with the archived distillation weight notebooks. The opening cells are organized so the editable experiment controls stay visible and close to the style of the original distillation notebooks.

In [ ]:
from pathlib import Path
import os

from utils.notebook_setup import prepare_distillation_notebook_env, print_notebook_summary

# Main notebook controls.
# These defaults are aligned with the archived distillation weight notebooks.
AGENT_KIND = "td3"  # "td3" | "sac"
RUN_MODE = "nominal"  # "nominal" | "disturb"
DISTURBANCE_PROFILE = "none"  # "none" | "ramp" | "fluctuation"
STATE_MODE = "standard"  # "standard" | "mismatch"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

# Aspen selection. Leave ASPEN_PRESET="default" to follow the archived family-specific plant file.
ASPEN_PRESET = "default"
ASPEN_PATH_OVERRIDE = None
SNAPS_PATH_OVERRIDE = None
ASPEN_ROOT_OVERRIDE = None
DISTILLATION_VISIBLE = True

DISTILLATION_DATA_DIR_OVERRIDE = None
DISTILLATION_RESULTS_DIR_OVERRIDE = None

# Optional naming/path overrides.
RESULT_PREFIX_OVERRIDE = None
COMPARE_PREFIX_OVERRIDE = None
BASELINE_MPC_PATH_OVERRIDE = None

# Optional run-size overrides. Leave as None to use the archived defaults.
N_TESTS_OVERRIDE = None
SET_POINTS_LEN_OVERRIDE = None
WARM_START_OVERRIDE = None
TEST_CYCLE_OVERRIDE = None
PLOT_START_EPISODE_OVERRIDE = None
COMPARE_START_EPISODE_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(
    run_mode=RUN_MODE,
    disturbance_profile=DISTURBANCE_PROFILE,
    family="weights",
    aspen_preset=ASPEN_PRESET,
    dyn_path_override=ASPEN_PATH_OVERRIDE,
    snaps_path_override=SNAPS_PATH_OVERRIDE,
    aspen_root_override=ASPEN_ROOT_OVERRIDE,
    data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE,
    results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

print_notebook_summary(
    "Distillation weights notebook configuration",
    {
        "Distillation data dir": DATA_DIR,
        "Distillation results": RESULT_DIR,
        "Agent kind": AGENT_KIND,
        "Run mode": RUN_MODE,
        "Disturbance profile": DISTURBANCE_PROFILE,
        "State mode": STATE_MODE,
        "Aspen source": ASPEN_SOURCE,
        "Aspen dynf": DYN_PATH,
        "Aspen snaps": SNAPS_PATH,
    },
)

import numpy as np
import torch

from SACAgent.sac_agent import SACAgent
from systems.distillation import DISTILLATION_SYSTEM_METADATA
from systems.distillation.config import (
    DELTA_T_HOURS,
    DISTILLATION_INPUT_BOUNDS,
    DISTILLATION_NOMINAL_CONDITIONS,
    DISTILLATION_OBSERVER_POLES,
    DISTILLATION_RL_SETPOINTS_PHYS,
    DISTILLATION_SETPOINT_RANGE_PHYS,
    DISTILLATION_SS_INPUTS,
    DISTILLATION_WEIGHT_RUN_PROFILES,
    RL_REWARD_DEFAULTS,
    WEIGHT_MULTIPLIER_BOUNDS,
)
from systems.distillation.data_io import canonical_baseline_path, load_distillation_system_data
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import build_distillation_disturbance_schedule
from Simulation.mpc import MpcSolverGeneral
from TD3Agent.agent import TD3Agent
from utils.helpers import apply_min_max
from utils.observer import compute_observer_gain
from utils.plotting import compare_mpc_rl_from_dirs, plot_weight_multiplier_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim
from utils.weights_runner import run_weight_multiplier_supervisor


In [ ]:
RUN_PROFILE = DISTILLATION_WEIGHT_RUN_PROFILES[(AGENT_KIND, RUN_MODE, DISTURBANCE_PROFILE)]


nominal_conditions = DISTILLATION_NOMINAL_CONDITIONS.copy()
ss_inputs = DISTILLATION_SS_INPUTS.copy()
u_min = DISTILLATION_INPUT_BOUNDS["u_min"].copy()
u_max = DISTILLATION_INPUT_BOUNDS["u_max"].copy()
setpoint_y = DISTILLATION_SETPOINT_RANGE_PHYS.copy()

# Legacy weight notebooks used the RL setpoint pair below.
y_sp_scenario_phys = DISTILLATION_RL_SETPOINTS_PHYS.copy()
system = build_distillation_system(
    path=DYN_PATH,
    ss_inputs=ss_inputs,
    initialization_point=nominal_conditions,
    delta_t=DELTA_T_HOURS,
    visible=DISTILLATION_VISIBLE,
)
steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
system_data = load_distillation_system_data(REPO_ROOT, steady_states, setpoint_y, u_min, u_max)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])

RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or f"distillation_weights_{AGENT_KIND}_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}_unified"
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or f"distillation_compare_weights_{AGENT_KIND}_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}"
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, DISTURBANCE_PROFILE)
n_tests = RUN_PROFILE["n_tests"] if N_TESTS_OVERRIDE is None else int(N_TESTS_OVERRIDE)
set_points_len = RUN_PROFILE["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else int(SET_POINTS_LEN_OVERRIDE)
warm_start = RUN_PROFILE["warm_start"] if WARM_START_OVERRIDE is None else int(WARM_START_OVERRIDE)
TEST_CYCLE = list(RUN_PROFILE["test_cycle"]) if TEST_CYCLE_OVERRIDE is None else list(TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else int(PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = RUN_PROFILE.get("compare_start_episode", PLOT_START_EPISODE) if COMPARE_START_EPISODE_OVERRIDE is None else int(COMPARE_START_EPISODE_OVERRIDE)
TOTAL_STEPS = n_tests * set_points_len * len(y_sp_scenario_phys)
DISTURBANCE_SCHEDULE = build_distillation_disturbance_schedule(RUN_MODE, DISTURBANCE_PROFILE, TOTAL_STEPS)


print_notebook_summary(
    "Resolved experiment parameters",
    {
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "warm_start": warm_start,
        "plot_start_episode": PLOT_START_EPISODE,
        "compare_start_episode": COMPARE_START_EPISODE,
        "legacy_weight_setpoints_phys": y_sp_scenario_phys.tolist(),
        "baseline_mpc_path": BASELINE_MPC_PATH,
    },
)


In [ ]:
# User-editable controller defaults. Edit these values directly when you want to change controller/agent behavior.
poles = DISTILLATION_OBSERVER_POLES.copy()
N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
STATE_DIM = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, STATE_MODE)
ACTION_DIM = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predict_h = 6
cont_h = 3
Q1_penalty = 1.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0
LOW_COEF = WEIGHT_MULTIPLIER_BOUNDS["low"].copy()
HIGH_COEF = WEIGHT_MULTIPLIER_BOUNDS["high"].copy()
ACTOR_FREEZE = 0

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty], float),
    R_in=np.array([R1_penalty, R2_penalty], float),
    NP=predict_h,
    NC=cont_h,
)

reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    N_INPUTS,
    **RL_REWARD_DEFAULTS,
)
print(reward_params)

nominal_qi = 0.0
nominal_qs = 0.0
nominal_hA = 0.0
qi_change = 1.0
qs_change = 1.0
ha_change = 1.0


if AGENT_KIND == "td3":
    weight_agent = TD3Agent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=[512, 512, 512, 512, 512],
        critic_hidden=[512, 512, 512, 512, 512],
        gamma=0.995,
        actor_lr=1e-4,
        critic_lr=1e-4,
        batch_size=128,
        policy_delay=2,
        target_policy_smoothing_noise_std=0.1,
        noise_clip=0.2,
        max_action=1.0,
        tau=0.005,
        std_start=0.2,
        std_end=0.02,
        std_decay_rate=0.99995,
        std_decay_mode="exp",
        buffer_size=50_000,
        device=DEVICE,
        actor_freeze=ACTOR_FREEZE,
    )
elif AGENT_KIND == "sac":
    weight_agent = SACAgent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=[512, 512, 512, 512, 512],
        critic_hidden=[512, 512, 512, 512, 512],
        gamma=0.995,
        actor_lr=1e-4,
        critic_lr=1e-4,
        alpha_lr=1e-4,
        batch_size=128,
        grad_clip_norm=10.0,
        init_alpha=0.01,
        learn_alpha=True,
        target_entropy=-ACTION_DIM,
        target_update="soft",
        tau=0.005,
        hard_update_interval=10_000,
        activation="relu",
        use_layernorm=False,
        dropout=0.0,
        max_action=1.0,
        buffer_size=50_000,
        use_per=True,
        device=DEVICE,
        use_adamw=True,
        actor_freeze=ACTOR_FREEZE,
    )
else:
    raise ValueError("AGENT_KIND must be 'td3' or 'sac'.")

print(f"State dimension: {STATE_DIM}")
print(weight_agent.__class__.__name__)


In [ ]:
weight_cfg = {
    "agent_kind": AGENT_KIND,
    "run_mode": RUN_MODE,
    "state_mode": STATE_MODE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "low_coef": LOW_COEF,
    "high_coef": HIGH_COEF,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
}

runtime_ctx = {
    "system": system,
    "agent": weight_agent,
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": DISTILLATION_SYSTEM_METADATA,
    "reward_params": reward_params,
    "system_stepper": distillation_system_stepper,
    "disturbance_labels": DISTILLATION_SYSTEM_METADATA.get("disturbance_labels"),
    "disturbance_schedule": DISTURBANCE_SCHEDULE,
}

result_bundle = run_weight_multiplier_supervisor(weight_cfg=weight_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH


In [ ]:
out_dir_rl = plot_weight_multiplier_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_MODE,
    start_episode=COMPARE_START_EPISODE,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print(out_dir_rl)
print(out_dir_cmp)

try:
    system.close(SNAPS_PATH)
except Exception:
    pass
